In [4]:
import pandas as pd
from os import path

# 'Name', 'Vessel Sanction Indicators', 'IMO', 'Flag', 'LLI Vessel Type', 'Vessel Type and Size', 'Built', 'Status', 'Beneficial Owner',
# 'Beneficial Owner Country/Region', 'Beneficial Owner Sanction Indicators', 'Beneficial Owner - Shareholders & UBO Sanctions',
# 'Commercial Operator', 'Commercial Operator Country/Region', 'Commercial Operator Sanction Indicators', 'Commercial Operator - Shareholders & UBO Sanctions',
# 'Registered Owner', 'Registered Owner Country/Region', 'Registered Owner Sanction Indicators', 'Registered Owner - Shareholders & UBO Sanctions', 'Technical Manager',
# 'Technical Manager Country/Region', 'Technical Manager Sanction Indicators', 'Technical Manager - Shareholders & UBO Sanctions', 'ISM Manager',
# 'ISM Manager Country/Region', 'ISM Manager Sanction Indicators', 'ISM Manager - Shareholders & UBO Sanctions', 'Third Party Operator', 'Third Party Operator Country/Region',
# 'Third Party Operator Sanction Indicators', 'Third Party Operator - Shareholders & UBO Sanctions', 'Length Overall (m)', 'Depth (m)', 'Maximum Draught (m)', 'Freeboard (m)'
vessels = pd.read_csv(path.join("data", "Vessels.csv"), sep=";")

# 'subjectid', 'timestamp', 'position', 'cog', 'sog', 'trueheading', 'navstatus', 'lng', 'lat'
radar_pos = pd.read_csv(path.join("data","AIS_RADAR_streams202260306", "radar_positionrecord_202603061623.csv"))

# 'vesselid', 'timestamp', 'position', 'cog', 'sog', 'trueheading', 'navstatus', 'lng', 'lat'
ais_pos = pd.read_csv(path.join("data","AIS_RADAR_streams202260306", "vessel_annotatedpositionrecord_202603061619.csv"))

print(f"vessels: {len(vessels)}, radar_pos: {len(radar_pos)}, vessel_ann: {len(ais_pos)}")

radar_pos[["lng", "lat"]] = radar_pos["position"].str.extract(r"POINT \(([^ ]+) ([^ ]+)\)").astype(float)
ais_pos[["lng", "lat"]] = ais_pos["position"].str.extract(r"POINT \(([^ ]+) ([^ ]+)\)").astype(float)

print("radar_pos", radar_pos["subjectid"].nunique(), radar_pos["timestamp"].min(), "->", radar_pos["timestamp"].max())
print("vessel_ann", ais_pos["vesselid"].nunique(), ais_pos["timestamp"].min(), "->", ais_pos["timestamp"].max())

vessels: 6151, radar_pos: 335382, vessel_ann: 3240214
radar_pos 2934 2026-03-06 13:33:00.388 -> 2026-03-06 14:33:59.896
vessel_ann 194932 2026-03-06 13:33:00.000 -> 2026-03-06 14:33:59.895


In [ ]:
import asyncio
from pyle38 import Tile38

t38 = Tile38(url="redis://localhost:9851")
BATCH_SIZE = 500

async def import_radar_tracks():
    rows = list(radar_pos.sort_values("timestamp").itertuples(index=False))
    total = len(radar_pos)
    for start in range(0, total, BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        await asyncio.gather(*[
            t38.set("radar_tracks", str(row.subjectid))
                   .fields({"sog": row.sog, "cog": row.cog, "timestamp": row.timestamp})
                   .point(row.lat, row.lng)
                   .exec()
                   for row in batch
               ])
        print(f"Imported {min(start + BATCH_SIZE, total)}/{total} positions")

async def import_ais_tracks():
    rows = list(ais_pos.sort_values("timestamp").itertuples(index=False))
    total = len(ais_pos)
    for start in range(0, total, BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        await asyncio.gather(*[
            t38.set("ais_tracks", str(row.vesselid))
                   .fields({"sog": row.sog, "cog": row.cog, "timestamp": row.timestamp})
                   .point(row.lat, row.lng)
                   .exec()
                   for row in batch
               ])
        print(f"Imported {min(start + BATCH_SIZE, total)}/{total} positions")

await import_radar_tracks()
await import_ais_tracks()
t38.quit()

Imported 500/335382 positions
Imported 1000/335382 positions
Imported 1500/335382 positions
Imported 2000/335382 positions
Imported 2500/335382 positions
Imported 3000/335382 positions
Imported 3500/335382 positions
Imported 4000/335382 positions
Imported 4500/335382 positions
Imported 5000/335382 positions
Imported 5500/335382 positions
Imported 6000/335382 positions
Imported 6500/335382 positions
Imported 7000/335382 positions
Imported 7500/335382 positions
Imported 8000/335382 positions
Imported 8500/335382 positions
Imported 9000/335382 positions
Imported 9500/335382 positions
Imported 10000/335382 positions
Imported 10500/335382 positions
Imported 11000/335382 positions
Imported 11500/335382 positions
Imported 12000/335382 positions
Imported 12500/335382 positions
Imported 13000/335382 positions
Imported 13500/335382 positions
Imported 14000/335382 positions
Imported 14500/335382 positions
Imported 15000/335382 positions
Imported 15500/335382 positions
Imported 16000/335382 positi

In [8]:
t38 = Tile38(url="redis://localhost:9851")
await t38.get("radar_tracks", "153207705").asObject()
t38.quit()

<coroutine object Tile38.quit at 0x135540520>